# 🎓 Multilingual Lecture-to-Notes & Quiz Generator
## CS3235 — CIE3 Lab Assessment

This notebook:
1. Imports the project from GitHub or an uploaded ZIP and installs dependencies
2. Reads `HF_TOKEN` from Colab Secrets
3. Runs the MOCK tests (model smoke tests are optional)
4. Starts the Gradio app in the background and prints a temporary public URL

> **Before running**: Zip the `lecture-notes-quiz` project folder (do not include `.env`), upload this notebook to Colab, and add `HF_TOKEN` in Colab Secrets. The share URL is temporary and works only while the Colab runtime stays connected.

> **Runtime**: Set to **GPU → T4** in Runtime → Change runtime type

In [ ]:
# ── Step 0: Check GPU ─────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠ No GPU detected — running on CPU (MOCK mode recommended)')

In [ ]:
# ── Step 1: Import project source ─────────────────────────────────────────
# Optional: set this to a public GitHub clone URL after publishing the repo.
REPO_URL = ''

import os
import zipfile
from pathlib import Path
from google.colab import files

PROJECT_DIR = Path('/content/lecture-notes-quiz')
if REPO_URL:
    !git clone {REPO_URL} /content/lecture-notes-quiz
else:
    print('Upload a ZIP of the lecture-notes-quiz project folder. Do not include .env or credentials.')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise RuntimeError('Upload one project ZIP file, or set REPO_URL to your GitHub repository.')
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_names[0]) as archive:
        destination = PROJECT_DIR.resolve()
        for member in archive.infolist():
            target = (PROJECT_DIR / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise RuntimeError(f'Unsafe path in project ZIP: {member.filename}')
        archive.extractall(PROJECT_DIR)
    app_files = list(PROJECT_DIR.rglob('app.py'))
    if not app_files:
        raise RuntimeError('Project ZIP must contain app.py and the src/ folder.')
    PROJECT_DIR = app_files[0].parent

os.chdir(PROJECT_DIR)
print('Project directory:', PROJECT_DIR)

In [ ]:
# ── Step 2: Install system dependencies ──────────────────────────────────
!apt-get install -qq ffmpeg

In [ ]:
# ── Step 3: Install Python dependencies ──────────────────────────────────
# Core requirements
!pip install -q -r requirements.txt

# IndicTransToolkit (not on PyPI, install from GitHub)
!pip install -q git+https://github.com/VarunGumma/IndicTransToolkit.git

# Replace CPU ONNX Runtime with the CUDA package for this T4 runtime.
!pip uninstall -y onnxruntime
!pip install -q onnxruntime-gpu==1.20.1 onnx==1.20.1

print('\n✅ Installation complete')

In [ ]:
# ── Step 4: Read Hugging Face token from Colab Secrets ─────────────────────
# Add HF_TOKEN in the Colab 🔑 Secrets panel. Never paste it into this notebook.
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN is missing. Add it in the Colab Secrets panel.')
os.environ['HF_TOKEN'] = hf_token
del hf_token
print('✅ HF_TOKEN loaded from Colab Secrets')

In [ ]:
# ── Step 5: Set real inference mode and T4 settings ──────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. In Colab choose Runtime → Change runtime type → T4 GPU, reconnect, then rerun.')
print('CUDA device:', torch.cuda.get_device_name(0))
os.environ['MOCK_MODELS'] = '0'

# Memory-efficient settings for T4 (15 GB VRAM):
os.environ['LLM_QUANTIZATION'] = 'int4'  # or 'int8' if int4 is unsupported

print(f"MOCK_MODELS={os.environ['MOCK_MODELS']}")
print(f"LLM_QUANTIZATION={os.environ['LLM_QUANTIZATION']}")

In [ ]:
# ── Step 6: Run pytest in MOCK mode ──────────────────────────────────────
!MOCK_MODELS=1 pytest tests/ -v --tb=short 2>&1 | tail -30

In [ ]:
# ── Optional: Run ASR smoke test on a sample clip ────────────────────────
# Leave False to get the web app URL without first waiting for model inference.
RUN_ASR_SMOKE_TEST = False

from pathlib import Path
if RUN_ASR_SMOKE_TEST:
    sample_files = [path for path in Path('data/audio').glob('*') if path.is_file()]
    sample_path = sample_files[0] if sample_files else None
    if sample_path is None:
        print('Upload a WAV, MP3, or M4A sample clip for ASR verification.')
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No sample clip was uploaded; ASR smoke test cannot run.')
        sample_path = Path(next(iter(uploaded)))
    from src.asr import transcribe
    from src.utils import load_config
    source_lang = load_config().get('default_language', 'hi')
    asr_result = transcribe(str(sample_path), lang=source_lang)
    print(asr_result['full_text'])
    print(f"ASR latency: {asr_result['latency_s']:.2f} s; chunks: {len(asr_result['chunks'])}")
    if not asr_result['full_text'].strip():
        raise RuntimeError('ASR smoke test completed with an empty transcript.')
else:
    print('ASR smoke test skipped. Upload audio to the running web app to transcribe it.')

In [ ]:
# ── Optional: Run a short transcript through notes + quiz ────────────────
# This loads Param-1. Leave False if you want to open the web app first.
RUN_NOTES_QUIZ_SMOKE_TEST = False

if RUN_NOTES_QUIZ_SMOKE_TEST:
    from src.notes import make_notes, free_llm_memory
    from src.quiz import make_mcqs
    from src.utils import load_config
    source_lang = load_config().get('default_language', 'hi')
    short_transcript = (
        'Newton\'s first law describes inertia. '
        'An object remains at rest unless an external force acts on it. '
        'Newton\'s second law relates force, mass, and acceleration. '
        'Newton\'s third law states that every action has an equal and opposite reaction. '
        'The lecture says these laws are used in rocket science and vehicle design.'
    )
    try:
        notes_result = make_notes(short_transcript, lang=source_lang)
        quiz_result = make_mcqs(notes_result, lang=source_lang, transcript=short_transcript)
        assert len(quiz_result.mcqs) == 5
        print(notes_result['markdown'])
        print(f"Notes + quiz smoke test passed: {len(quiz_result.mcqs)} MCQs")
    finally:
        free_llm_memory()
else:
    print('Notes + quiz smoke test skipped. It can be enabled after the app is running.')

In [ ]:
# ── Step 9: Start the web app and print its temporary public URL ─────────
# Keep the Colab runtime connected while you or others use this URL.
import re
import subprocess
import sys
import time

log_path = Path('/content/lecture-notes-quiz-gradio.log')
log_file = open(log_path, 'w', encoding='utf-8')
app_process = subprocess.Popen(
    [sys.executable, '-u', 'app.py', '--share'],
    cwd=PROJECT_DIR,
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

deadline = time.time() + 180
share_url = None
while time.time() < deadline:
    log_file.flush()
    log_text = log_path.read_text(encoding='utf-8', errors='replace')
    match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', log_text)
    if match:
        share_url = match.group(0)
        break
    if app_process.poll() is not None:
        raise RuntimeError(f'Gradio exited with code {app_process.returncode}. Last log lines:\n{log_text[-4000:]}')
    time.sleep(2)

if not share_url:
    tail = log_path.read_text(encoding='utf-8', errors='replace')[-4000:]
    raise TimeoutError(f'Gradio did not print a share URL within 180 seconds. Log tail:\n{tail}')

print('Temporary web app URL:', share_url)
print('The URL is temporary. Keep this Colab runtime connected; stopping it shuts down the app.')
print('Server log:', log_path)

## Evaluation (optional)

After generating outputs for all test cases, run the evaluation:

In [ ]:
# Run full evaluation suite
!python eval/run_all.py

In [ ]:
# View generated charts
from IPython.display import Image, display
from pathlib import Path

for fig in Path('results/figures').glob('*.png'):
    print(fig.name)
    display(Image(str(fig)))